# 37 — Classical image processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/ai-dl/37-image-processing/lab.ipynb) [![Open in LinguAgent](https://img.shields.io/badge/Open%20in-LinguAgent-1f6f5c)](https://linguagent.com/courses/ai-dl/image-processing?utm_source=github&utm_medium=labs)

*~50 minutes · 19 tests*

Before a learned convolution ever sees a pixel, decades of hand-designed
filters already solved blurring, edge detection, contrast enhancement and
resizing — and those techniques still show up constantly, as
preprocessing, as augmentation, and as the vocabulary an interviewer uses
to ask about them. This module builds the classical toolkit: Gaussian
blur, the Sobel edge detector, histogram equalization, and bilinear
resizing — checked against PyTorch's own `F.conv2d` and `F.interpolate`
where an equivalent exists.

Work in `exercises.py`.

**How this notebook works:** run the first cell; then, section by section, read, write the code in the cell under it, and run the check below it. Stuck? `lab.solution("name")` shows the reference — try first, it's worth it.

In [ ]:
#@title ▶️ Run me first — fetches this lab and its checker
import os, sys, subprocess
HOME = "/content/linguagent-labs"
if "google.colab" in sys.modules:
    if not os.path.isdir(HOME):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/mrshahabian/linguagent-labs.git", HOME], check=True)
else:  # Jupyter, VS Code or a Codespace: this notebook already sits inside the repo
    HOME = os.path.abspath(os.path.join(os.getcwd(), *[".."] * 3)) if os.path.isdir("../../../labkit") else os.getcwd()
sys.path.insert(0, HOME)
import labkit
lab = labkit.open("ai-dl/image-processing")

In [ ]:
# What the exercises below use — run this cell.
import torch

import torch.nn.functional as F

## What you'll build

| Name | You'll implement | Shows up in real code as |
|---|---|---|
| `gaussian_kernel` | the function | The blur kernel behind noise reduction and scale-space methods |
| `apply_kernel` | the function | "Same"-padding convolution — the mechanics under every classical filter |
| `sobel_edges` | the function | The pre-deep-learning edge detector, still used as a preprocessing step |
| `histogram_equalization` | the function | Contrast enhancement — the standard fix for a washed-out or dark image |
| `bilinear_resize` | the function | The interpolation behind every image resize that isn't nearest-neighbour |

---

## 1. A Gaussian kernel, and applying it

```python
kernel = torch.exp(-(grid_x ** 2 + grid_y ** 2) / (2 * sigma ** 2))
kernel = kernel / kernel.sum()
```

A 2D Gaussian kernel, normalised so its values sum to exactly 1 —
normalising is what keeps a blurred image's overall brightness
unchanged; an unnormalised kernel would systematically darken or
brighten the result depending on whether its sum is below or above 1.
(Real implementations exploit that a 2D Gaussian is *separable* — the
outer product of two 1D Gaussians — and convolve rows then columns
instead of building the full 2D kernel, purely for speed; this exercise
builds the 2D kernel directly since the shape is the point.)

```python
def apply_kernel(image, kernel):
    padding = kernel.shape[-1] // 2
    return F.conv2d(image[None, None], kernel[None, None], padding=padding).squeeze()
```

Padding by `kernel_size // 2` before convolving is what makes the output
the same size as the input — a "same" convolution, using the identical
padding arithmetic as `conv_output_size` back in module 27, applied here
so a filtered image doesn't shrink every time you apply one.

```python
>>> kernel = gaussian_kernel(5, sigma=1.0)
>>> kernel.sum()
tensor(1.0000)
>>> apply_kernel(image, kernel).shape == image.shape
True
```

**In `exercises.py`:** `gaussian_kernel`, `apply_kernel`.

In [ ]:
def gaussian_kernel(size, sigma):
    """A (size, size) 2D Gaussian kernel, normalised to sum to 1.

    Separable in principle (a 2D Gaussian is the outer product of two 1D
    Gaussians, which is why real implementations convolve rows then
    columns instead of doing the full 2D kernel) - built directly here as
    a 2D grid for clarity, since the point of this exercise is the shape
    of the kernel, not the factorisation that makes it fast.

    Normalising to sum 1 is what keeps a blurred image's overall
    brightness unchanged - an unnormalised kernel would systematically
    darken or brighten the image depending on whether its sum is below
    or above 1.
    """
    raise NotImplementedError("your code here")

In [ ]:
def apply_kernel(image, kernel):
    """Convolve a single-channel image with a 2D kernel, same output size.

    Padding by kernel_size // 2 on each side before F.conv2d's default
    "valid" convolution is what makes output size equal input size (a
    "same" convolution) - the same padding arithmetic as module 27's
    conv_output_size, applied here to make classical filters preserve
    image dimensions the way every real image-filtering function does.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§1", globals())

## 2. Sobel: edges as a gradient magnitude

```python
gx_kernel = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]
gy_kernel = [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]
magnitude = sqrt(gx ** 2 + gy ** 2)
```

Two fixed 3x3 kernels approximate the horizontal and vertical intensity
gradient at every pixel. Combining them as a gradient **magnitude**
(rather than looking at `Gx` or `Gy` alone) responds to an edge
regardless of its orientation — a horizontal edge, a vertical edge, and
a diagonal edge all produce a large magnitude, just from different
combinations of `Gx` and `Gy`. This is the pre-deep-learning answer to
"where are the edges in this image": a fixed, hand-designed filter,
where a CNN's early layers instead *learn* whatever edge-like filters
turn out to be useful for the task.

**A real gotcha worth knowing:** `apply_kernel`'s zero-padding invents a
false intensity drop right at the image border (a real pixel sitting
next to a padded zero), so even a perfectly flat image shows a nonzero
"edge" exactly on its border — genuinely zero only in the interior.
Every edge detector either accepts this border artifact or pads
differently (replicating the edge pixel, mirroring the image) to avoid
it.

```python
>>> flat = torch.ones(8, 8) * 100
>>> sobel_edges(flat)[1:-1, 1:-1]   # interior: exactly zero
tensor([[0., 0., ...
```

**In `exercises.py`:** `sobel_edges`.

In [ ]:
def sobel_edges(image):
    """Gradient magnitude via the Sobel operator - the classic edge detector.

    Two 3x3 kernels approximate the horizontal and vertical intensity
    gradient (Gx, Gy) at each pixel; combining them as
    sqrt(Gx^2 + Gy^2) gives the gradient MAGNITUDE, which is large at an
    edge (a sharp intensity change in either direction) regardless of the
    edge's orientation. This is the pre-deep-learning answer to "where
    are the edges in this image" - a fixed, hand-designed filter rather
    than a learned one.

    Note apply_kernel's zero-padding invents a fake intensity drop at the
    image border (real pixels next to padded zeros), so even a perfectly
    flat image shows a false "edge" exactly on its border - genuinely
    zero only in the interior. Every real edge detector has to either
    accept this border artifact or pad some other way (replicate the
    edge pixel, mirror the image) to avoid it.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§2", globals())

## 3. Histogram equalization: stretching a narrow band of intensities

```python
cdf = torch.cumsum(histogram, dim=0)
cdf_min = cdf[cdf > 0][0]
cdf_normalized = (cdf - cdf_min) / (total - cdf_min)
lookup_table = (cdf_normalized * 255).clamp(min=0).round().long()
```

The cumulative histogram (CDF) at intensity `v` is "what fraction of all
pixels are at or below `v`." Remapping each pixel through its own CDF
stretches a low-contrast image — one whose intensities are bunched into
a narrow band — across the *full* output range, because the CDF rises
steeply through that narrow band: the same amount of "probability mass"
now spans more output values.

**Subtracting `cdf_min`** — the CDF at the image's own darkest present
value — is what forces the darkest pixel to map to exactly 0 and the
brightest to exactly 255, matching the standard formula used by
OpenCV's `equalizeHist`. Skip that subtraction and the darkest pixel
maps to some value *above* zero, since its own count is already folded
into `cdf(v)` at `v` itself.

```python
>>> histogram_equalization(torch.tensor([[0, 0], [255, 255]]))
tensor([[  0,   0],
        [255, 255]])
```

**In `exercises.py`:** `histogram_equalization`.

In [ ]:
def histogram_equalization(image):
    """Redistribute a grayscale image's intensities to use the full 0-255 range.

    Built from the image's own cumulative histogram (CDF): a pixel's new
    value is proportional to what fraction of all pixels are at or below
    its current value. An image whose intensities are bunched into a
    narrow band (low contrast) gets that band stretched across the full
    range, because the CDF rises steeply there - the same amount of
    "probability mass" now spans more output values.

    Subtracting cdf_min (the CDF at the image's own darkest present
    value) before normalising is what forces the darkest pixel in the
    image to map to exactly 0, and the brightest to exactly 255 - the
    standard formula (matching OpenCV's equalizeHist), not just "roughly
    stretched." Without that subtraction the darkest pixel would map to
    some value above 0 instead, since its own count is already folded
    into cdf(v) at v itself.

    image: integer tensor, values in [0, 255].
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§3", globals())

## 4. Bilinear resize: blending four neighbours, not picking one

```python
src_y = (out_y + 0.5) * scale_y - 0.5
y0, y1 = int(src_y), min(int(src_y) + 1, in_height - 1)
wy = src_y - y0
# blend the four (y0,x0), (y0,x1), (y1,x0), (y1,x1) neighbours by wx, wy
```

Nearest-neighbour resizing just picks whichever source pixel is
closest, which produces visible blockiness scaling up. Bilinear instead
maps each output pixel back to a **fractional** source coordinate and
blends the four neighbouring source pixels, weighted by how close that
fractional coordinate is to each one. The `+ 0.5` / `- 0.5` shift is
what aligns pixel *centers* rather than pixel corners between the two
grids — the same convention `F.interpolate(..., align_corners=False)`
uses, which is why this exercise is checked against it directly rather
than against a hand-derived expected array.

```python
>>> bilinear_resize(image, out_height=12, out_width=16)
# matches F.interpolate(image, size=(12, 16), mode="bilinear", align_corners=False)
```

**In `exercises.py`:** `bilinear_resize`.

In [ ]:
def bilinear_resize(image, out_height, out_width):
    """Resize a single-channel image using bilinear interpolation.

    Nearest-neighbour resizing just picks the closest source pixel,
    which produces visible blockiness when scaling up. Bilinear instead
    maps each output pixel back to a (fractional) source coordinate and
    blends the FOUR neighbouring source pixels, weighted by how close
    the fractional coordinate is to each one - the same shape of
    interpolation `F.interpolate(mode="bilinear")` does, built here from
    the coordinate mapping up.
    """
    raise NotImplementedError("your code here")

In [ ]:
lab.check("§4", globals())

---

## The exercises

| Function | What it teaches |
|---|---|
| `gaussian_kernel` | Normalisation preserves brightness; the shape of a blur |
| `apply_kernel` | "Same"-padding convolution mechanics |
| `sobel_edges` | Gradient magnitude as orientation-independent edge strength |
| `histogram_equalization` | CDF-based contrast stretching, and why `cdf_min` matters |
| `bilinear_resize` | Fractional-coordinate interpolation, pixel-center alignment |

Every function in this table has a stub in `exercises.py` and test
coverage in `test_exercises.py` — a clean 1:1 mapping.

## Check your understanding

1. **Why does a Gaussian kernel need to be normalised to sum to 1
   before you convolve an image with it?**
   <details><summary>Answer</summary>Convolution's output at each pixel
   is a weighted sum of neighbouring pixel values using the kernel's
   entries as weights. If those weights don't sum to 1, the output is
   systematically scaled up or down relative to the input's brightness —
   normalising to 1 is what keeps a blurred image's overall brightness
   matching the original.</details>

2. **Why does Sobel combine `Gx` and `Gy` as `sqrt(Gx² + Gy²)` instead of
   just using `Gx` (or `Gy`) alone?**
   <details><summary>Answer</summary><code>Gx</code> alone only responds
   strongly to vertical edges (horizontal intensity change), and
   <code>Gy</code> alone only to horizontal edges. Combining them as a
   magnitude responds to an edge of ANY orientation — a diagonal edge
   produces moderate values in both <code>Gx</code> and <code>Gy</code>
   that combine into a large magnitude, even though neither alone would
   flag it strongly.</details>

3. **Why does a perfectly flat, constant-value image still show a
   nonzero Sobel response at its border?**
   <details><summary>Answer</summary>Convolving with padding adds zeros
   just outside the image, and a border pixel's neighbourhood includes
   those zeros right next to real (nonzero) pixel values — that contrast
   is a real intensity jump from the filter's point of view, even though
   it's an artifact of the padding, not a real edge in the
   image.</details>

4. **What does subtracting `cdf_min` in histogram equalization actually
   fix?**
   <details><summary>Answer</summary>Without it, the image's darkest
   present intensity would map to some value above 0, because its own
   pixel count is already included in the CDF at that intensity.
   Subtracting the CDF's value at the darkest present intensity resets
   that floor to exactly 0, so the full output range is genuinely used
   from 0 to 255, matching the standard formula (and OpenCV's
   `equalizeHist`).</details>

5. **What visual artifact does nearest-neighbour resizing produce that
   bilinear resizing avoids, and why?**
   <details><summary>Answer</summary>Blockiness — nearest-neighbour
   assigns each output pixel the value of whichever single source pixel
   is closest, so scaling up produces visible flat blocks where several
   output pixels copy the same source value. Bilinear blends four
   neighbouring source pixels by how close the mapped fractional
   coordinate is to each, producing a smooth gradient between source
   pixels instead of a hard step.</details>

## Interview questions this module answers

- *Why do we normalise a blur kernel before applying it?*
- *How does Sobel edge detection work, and why combine two kernels
  instead of one?*
- *An image looks washed out or too dark — how would you fix its
  contrast programmatically?*
- *What's the difference between nearest-neighbour and bilinear image
  resizing?*

## Finished? Check everything, then record it

The first cell checks the whole unit. The second records your result in LinguAgent (needs your lab token from [https://linguagent.com/settings#labs](https://linguagent.com/settings#labs) — in Colab, add it once as a secret named `LINGUAGENT_TOKEN` under 🔑 in the left bar).

In [ ]:
lab.check_all(globals())

In [ ]:
lab.sync()

**Next:** [DL 15 - Binarized and quantized networks](https://colab.research.google.com/github/mrshahabian/linguagent-labs/blob/main/units/ai-dl/38-binarized-networks/lab.ipynb) · [the lesson in LinguAgent](https://linguagent.com/courses/ai-dl/image-processing)